In [ ]:
%pip install konlpy

text = "어제는 친구와 함께 경복궁에 방문했는데, 날씨가 너무 맑아서 기분이 정말 좋았어요. 한복을 빌려 입고 고궁을 산책하며 사진을 찍으니 마치 조선 시대로 시간 여행을 떠난 것 같은 기분이 들더라고요! 외국인 관광객들도 한국의 전통미에 감탄하며 즐거워하는 모습이 인상 깊었습니다. 다음에도 기회가 된다면 가족들과 꼭 다시 방문하고 싶네요. #서울여행 #경복궁 #전통문화 #힐링"

from konlpy.tag import Okt

o = Okt() # 오픈 한국어 텍스트 분석기 (구 트위터 한글 형태소 분석기)

a = o.normalize(text) # 정규화 (=정리)
print(a)

b = o.phrases(text) # 어구 추출 (=의미 기반 분리)
print(b)

c = o.morphs(text) # 형태소 분석 (= 단어보다 더 작은 최소 단위로)
print(c)

d = o.morphs(text, stem=True) # 어근 / 원형으로까지 분석 [c.f) 맑아서 => 맑다]
print(d)

e = o.pos(text) # + 품사 태깅
for word, wordtype in e:
    print(word, wordtype)

In [ ]:
%pip install jamo
from jamo import h2j, j2hcj

# 1) h2j: 음절을 표준 자모 형태로 분리
# '한' -> 'ㅎ', 'ㅏ', 'ㄴ' (표준 유니코드 자모 조합)
text = "그 남다른 마법사의 절친, 커트 폐하"
jamo_text = h2j(text)
print(f"원문: {text}")
print(f"h2j 결과: {jamo_text}") # 외관상으로는 차이가 없으나 내부 데이터 구조가 분리됨

# 2.)j2hcj: 분리된 자모를 가독성 있는 현대 자음/모음으로 변환
# 분석이나 학습 데이터 전처리 시 주로 사용
jamo_readable = j2hcj(jamo_text)
print(f"자모 분리 결과: {jamo_readable}") 

# 3. 활용 예시: 특정 자음 포함 여부 확인
target = "ㄱ"
if target in jamo_readable:
    print(f"문장에 '{target}'이(가) 포함되어 있습니다.")

In [ ]:
from http.client import HTTPSConnection
from bs4 import BeautifulSoup
import os

# 1. 경로 설정
target_dir = "./Users/student02_04/Jan23_1_Text"
os.makedirs(target_dir, exist_ok=True)

# 2. 서버 연결 및 데이터 요청
# 캡처본에 나타난 실제 브라우저의 User-Agent를 반영하여 차단을 방지합니다.
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36"
}

hc = HTTPSConnection("news.naver.com")
hc.request("GET", "/", headers=headers)

res = hc.getresponse()
if res.status != 200:
    print(f"연결 실패: HTTP {res.status}")
    hc.close()
else:
    res_body = res.read()
    hc.close()

    # 3. BeautifulSoup 파싱
    soup = BeautifulSoup(res_body, "html.parser")
    
    # 캡처본 분석 결과에 따른 선택자 설정
    # 메인 헤드라인: <strong class="cnf_news_title">
    # 하단 리스트: <li class="cnf_news_item"> 내부의 <a> 태그
    titles = soup.select("strong.cnf_news_title, li.cnf_news_item a")

    # 4. 파일 저장
    full_path = os.path.join(target_dir, "naver_news_titles.txt")
    
    with open(full_path, "w", encoding="utf-8") as f:
        f.write("=== 네이버 뉴스 실시간 헤드라인 ===\n")
        
        count = 0
        for title in titles:
            clean_title = title.get_text().strip()
            # 의미 없는 짧은 텍스트나 빈 줄 제외
            if len(clean_title) > 5:
                count += 1
                f.write(f"{count}. {clean_title}\n")

    print(f"성공: {count}개의 뉴스 제목을 '{full_path}'에 저장하였습니다.")

In [ ]:
# 동사 분석

f = open("./Users/student02_04/Jan23_1_Text/naver_news_titles.txt", "r", encoding="utf-8")

wc = {}
verbs = ""

for line in f.readlines():
    line = line.replace("\n", "")
    line = o.normalize(line)
    for w, p in o.pos(line, stem = True):
        if p == "Verb":
            verbs += w + " "
            if w in wc:
                wc[w] += 1
            else:
                wc[w] = 1
f.close()

print(wc)

In [ ]:
# 결과 시각화

import matplotlib.pyplot as plt
import numpy as np
import matplotlib.font_manager as fm

# 1. 폰트 경로 및 속성 설정
font_path = "./Users/student02_04/Jan23_1_Text/MALGUN.TTF"
# 폰트 속성 객체를 직접 생성합니다.
font_prop = fm.FontProperties(fname=font_path)
font_name = font_prop.get_name()

# 2. 데이터 준비 (상위 30개)
sorted_wc_list = sorted(wc.items(), key=lambda x: x[1], reverse=True)[:30]
words = [item[0] for item in sorted_wc_list]
counts = [item[1] for item in sorted_wc_list]

# 3. 시각화 설정
x_pos = np.arange(len(words))
plt.figure(figsize=(15, 8)) # 가독성을 위해 가로 폭을 넓혔습니다.
plt.bar(x_pos, counts, align='center', color='skyblue', edgecolor='navy')

# 4. 축 및 제목 설정 (fontproperties 매개변수 사용)
# 한글이 포함된 모든 요소에 font_prop을 직접 전달합니다.
plt.xticks(x_pos, words, rotation=45, fontproperties=font_prop)
plt.xlabel('Verbs', fontproperties=font_prop)
plt.ylabel('Frequency', fontproperties=font_prop)
plt.title('Top 30 Verbs Frequency', fontproperties=font_prop, fontsize=15)

# 5. 수치 표시
for i, v in enumerate(counts):
    plt.text(i, v + 0.1, str(v), ha='center', fontweight='bold')

plt.tight_layout()

# 6. 파일 저장 (Azure 환경에서 이미지가 안 뜰 경우 대비)
save_path = "./Users/student02_04/Jan23_1_Text/verb_frequency.png"
plt.savefig(save_path)
plt.show()